In [2]:
# ================================================================
# 🚀 FUTURE INTERNS - DATA SCIENCE & ANALYTICS
# TASK 3 - MARKETING FUNNEL & CONVERSION PERFORMANCE ANALYSIS
# 🌈 COMPLETE COLORFUL INTERACTIVE DASHBOARD
# ================================================================

# -------------------- 1. IMPORT LIBRARIES -----------------------

import pandas as pd
import numpy as np
import requests
import zipfile
import io
import os
import glob

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

print("🚀 Starting Future Interns Task 3...")


# -------------------- 2. DOWNLOAD DATASET ----------------------

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"

response = requests.get(url)

if response.status_code != 200:
    raise Exception("❌ Dataset download failed.")

print("✅ Dataset downloaded successfully!")


# -------------------- 3. EXTRACT DATASET ------------------------

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("future_interns_task3_data")

print("✅ Dataset extracted successfully!")


# -------------------- 4. FIND CSV FILE --------------------------

csv_files = glob.glob(
    "future_interns_task3_data/**/*.csv",
    recursive=True
)

print("\n📂 CSV files found:")

for file in csv_files:
    print("   ", file)

bank_file = None

for file in csv_files:
    if os.path.basename(file).lower() == "bank-full.csv":
        bank_file = file
        break

if bank_file is None:
    raise FileNotFoundError(
        "❌ bank-full.csv was not found."
    )

print("\n✅ Dataset selected:")
print(bank_file)


# -------------------- 5. LOAD DATA ------------------------------

df = pd.read_csv(
    bank_file,
    sep=";"
)

print("\n✅ Dataset loaded successfully!")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# -------------------- 6. CLEAN DATA -----------------------------

df = df.drop_duplicates().copy()

# Conversion variable
df["conversion"] = df["y"].map({
    "yes": 1,
    "no": 0
})

# Conversion status
df["conversion_status"] = df["y"].map({
    "yes": "Converted",
    "no": "Not Converted"
})

# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "66+"
    ],
    include_lowest=True
)

print("✅ Data cleaning completed!")


# ================================================================
# 7. CALCULATE MAIN KPIs
# ================================================================

total_contacts = len(df)

total_conversions = int(
    df["conversion"].sum()
)

total_dropoffs = (
    total_contacts - total_conversions
)

conversion_rate = (
    total_conversions / total_contacts
) * 100

dropoff_rate = (
    total_dropoffs / total_contacts
) * 100


# ================================================================
# 8. COLOR PALETTE
# ================================================================

BLUE = "#2563EB"
PURPLE = "#7C3AED"
PINK = "#EC4899"
ORANGE = "#F97316"
GREEN = "#10B981"
CYAN = "#06B6D4"
RED = "#EF4444"
YELLOW = "#F59E0B"

BACKGROUND = "#EEF2FF"
DARK = "#172033"


# ================================================================
# 9. CHANNEL ANALYSIS
# ================================================================

channel_data = (
    df.groupby("contact")["conversion"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


# ================================================================
# 10. AGE ANALYSIS
# ================================================================

age_data = (
    df.groupby(
        "age_group",
        observed=True
    )["conversion"]
    .mean()
    .mul(100)
)


# ================================================================
# 11. JOB ANALYSIS
# ================================================================

job_data = (
    df.groupby("job")["conversion"]
    .mean()
    .mul(100)
    .sort_values(ascending=True)
)


# ================================================================
# 12. EDUCATION ANALYSIS
# ================================================================

education_data = (
    df.groupby("education")["conversion"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


# ================================================================
# 13. MONTH ANALYSIS
# ================================================================

month_order = [
    "jan", "feb", "mar", "apr",
    "may", "jun", "jul", "aug",
    "sep", "oct", "nov", "dec"
]

month_data = (
    df.groupby("month")["conversion"]
    .mean()
    .mul(100)
    .reindex(month_order)
)


# ================================================================
# 14. PREVIOUS CAMPAIGN ANALYSIS
# ================================================================

previous_data = (
    df.groupby("poutcome")["conversion"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


# ================================================================
# 15. CREATE DASHBOARD
# ================================================================

fig = make_subplots(

    rows=6,
    cols=2,

    specs=[

        # Row 1
        [
            {"type": "indicator"},
            {"type": "indicator"}
        ],

        # Row 2
        [
            {"type": "indicator"},
            {"type": "indicator"}
        ],

        # Row 3
        [
            {"type": "funnel"},
            {"type": "bar"}
        ],

        # Row 4
        [
            {"type": "bar"},
            {"type": "bar"}
        ],

        # Row 5
        [
            {"type": "bar"},
            {"type": "bar"}
        ],

        # Row 6
        [
            {"type": "bar"},
            {"type": "bar"}
        ]
    ],

    vertical_spacing=0.075,

    subplot_titles=[

        "👥 Total Marketing Contacts",
        "🎯 Successful Conversions",

        "📈 Conversion Rate",
        "📉 Drop-off Rate",

        "🔻 Marketing Funnel",
        "📱 Conversion by Contact Channel",

        "👤 Conversion by Age Group",
        "💼 Conversion by Job",

        "📚 Conversion by Education",
        "📅 Conversion by Month",

        "🔄 Previous Campaign Outcome",
        "📊 Conversion Overview"
    ]
)


# ================================================================
# KPI 1
# ================================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=total_contacts,

        number={
            "font": {
                "size": 42,
                "color": BLUE
            },
            "valueformat": ","
        },

        title={
            "text": "<b>Total Contacts</b>"
        }
    ),

    row=1,
    col=1
)


# ================================================================
# KPI 2
# ================================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=total_conversions,

        number={
            "font": {
                "size": 42,
                "color": GREEN
            },
            "valueformat": ","
        },

        title={
            "text": "<b>Successful Conversions</b>"
        }
    ),

    row=1,
    col=2
)


# ================================================================
# KPI 3
# ================================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=conversion_rate,

        number={
            "suffix": "%",

            "font": {
                "size": 42,
                "color": PURPLE
            },

            "valueformat": ".2f"
        },

        title={
            "text": "<b>Conversion Rate</b>"
        }
    ),

    row=2,
    col=1
)


# ================================================================
# KPI 4
# ================================================================

fig.add_trace(

    go.Indicator(

        mode="number",

        value=dropoff_rate,

        number={
            "suffix": "%",

            "font": {
                "size": 42,
                "color": RED
            },

            "valueformat": ".2f"
        },

        title={
            "text": "<b>Drop-off Rate</b>"
        }
    ),

    row=2,
    col=2
)


# ================================================================
# FUNNEL CHART
# ================================================================

fig.add_trace(

    go.Funnel(

        y=[
            "Marketing Contacts",
            "Successful Conversions"
        ],

        x=[
            total_contacts,
            total_conversions
        ],

        textinfo="value+percent initial",

        marker={
            "color": [
                BLUE,
                GREEN
            ]
        },

        connector={
            "line": {
                "color": "#CBD5E1"
            }
        }
    ),

    row=3,
    col=1
)


# ================================================================
# CONTACT CHANNEL
# ================================================================

fig.add_trace(

    go.Bar(

        x=channel_data.index,

        y=channel_data.values,

        marker={
            "color": [
                CYAN,
                PURPLE,
                ORANGE
            ]
        },

        text=[
            f"{x:.1f}%"
            for x in channel_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Conversion Rate: %{y:.2f}%<extra></extra>"
    ),

    row=3,
    col=2
)


# ================================================================
# AGE GROUP
# ================================================================

fig.add_trace(

    go.Bar(

        x=age_data.index.astype(str),

        y=age_data.values,

        marker={
            "color": [
                PINK,
                PURPLE,
                BLUE,
                CYAN,
                GREEN,
                ORANGE
            ]
        },

        text=[
            f"{x:.1f}%"
            for x in age_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>Age Group: %{x}</b><br>" +
        "Conversion: %{y:.2f}%<extra></extra>"
    ),

    row=4,
    col=1
)


# ================================================================
# JOB
# ================================================================

fig.add_trace(

    go.Bar(

        x=job_data.values,

        y=job_data.index,

        orientation="h",

        marker={
            "color": PURPLE
        },

        text=[
            f"{x:.1f}%"
            for x in job_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{y}</b><br>" +
        "Conversion: %{x:.2f}%<extra></extra>"
    ),

    row=4,
    col=2
)


# ================================================================
# EDUCATION
# ================================================================

fig.add_trace(

    go.Bar(

        x=education_data.index,

        y=education_data.values,

        marker={
            "color": ORANGE
        },

        text=[
            f"{x:.1f}%"
            for x in education_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Conversion: %{y:.2f}%<extra></extra>"
    ),

    row=5,
    col=1
)


# ================================================================
# MONTH
# ================================================================

fig.add_trace(

    go.Bar(

        x=month_data.index,

        y=month_data.values,

        marker={
            "color": PINK
        },

        text=[
            f"{x:.1f}%"
            if pd.notna(x)
            else ""
            for x in month_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Conversion: %{y:.2f}%<extra></extra>"
    ),

    row=5,
    col=2
)


# ================================================================
# PREVIOUS CAMPAIGN
# ================================================================

fig.add_trace(

    go.Bar(

        x=previous_data.index,

        y=previous_data.values,

        marker={
            "color": GREEN
        },

        text=[
            f"{x:.1f}%"
            for x in previous_data.values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Conversion: %{y:.2f}%<extra></extra>"
    ),

    row=6,
    col=1
)


# ================================================================
# CONVERSION OVERVIEW
# ================================================================

overview_labels = [
    "Converted",
    "Not Converted"
]

overview_values = [
    total_conversions,
    total_dropoffs
]

fig.add_trace(

    go.Bar(

        x=overview_labels,

        y=overview_values,

        marker={
            "color": [
                GREEN,
                RED
            ]
        },

        text=[
            f"{x:,}"
            for x in overview_values
        ],

        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>" +
        "Customers: %{y:,}<extra></extra>"
    ),

    row=6,
    col=2
)


# ================================================================
# DASHBOARD DESIGN
# ================================================================

fig.update_layout(

    title={

        "text":
        "🚀 <b>MARKETING FUNNEL & CONVERSION PERFORMANCE</b>"
        "<br>"
        "<span style='font-size:17px'>"
        "Future Interns • Data Science & Analytics • Task 3"
        "</span>",

        "x": 0.5,

        "xanchor": "center"
    },

    title_font={
        "size": 28,
        "color": DARK
    },

    height=2200,

    width=1350,

    paper_bgcolor=BACKGROUND,

    plot_bgcolor="white",

    font={
        "family": "Arial",
        "color": DARK
    },

    showlegend=False,

    margin={
        "l": 90,
        "r": 90,
        "t": 160,
        "b": 80
    }
)


# ================================================================
# AXIS DESIGN
# ================================================================

fig.update_xaxes(
    showgrid=False
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#E5E7EB"
)


# Percentage axes

fig.update_yaxes(
    ticksuffix="%",
    row=3,
    col=2
)

fig.update_yaxes(
    ticksuffix="%",
    row=4,
    col=1
)

fig.update_xaxes(
    ticksuffix="%",
    row=4,
    col=2
)

fig.update_yaxes(
    ticksuffix="%",
    row=5,
    col=1
)

fig.update_yaxes(
    ticksuffix="%",
    row=5,
    col=2
)

fig.update_yaxes(
    ticksuffix="%",
    row=6,
    col=1
)


# ================================================================
# SHOW DASHBOARD
# ================================================================

fig.show()


# ================================================================
# KEY INSIGHTS
# ================================================================

best_channel = channel_data.idxmax()
best_channel_rate = channel_data.max()

best_job = job_data.idxmax()
best_job_rate = job_data.max()

best_age = age_data.idxmax()
best_age_rate = age_data.max()

best_education = education_data.idxmax()
best_education_rate = education_data.max()

best_previous = previous_data.idxmax()
best_previous_rate = previous_data.max()


print("\n")
print("=" * 80)
print("📊 FUTURE INTERNS TASK 3 - KEY INSIGHTS")
print("=" * 80)

print(
    f"\n👥 Total Marketing Contacts : {total_contacts:,}"
)

print(
    f"🎯 Successful Conversions   : {total_conversions:,}"
)

print(
    f"📈 Conversion Rate          : {conversion_rate:.2f}%"
)

print(
    f"📉 Drop-off Rate            : {dropoff_rate:.2f}%"
)

print("\n🏆 TOP PERFORMING SEGMENTS")
print("-" * 80)

print(
    f"📱 Contact Channel : "
    f"{best_channel} ({best_channel_rate:.2f}%)"
)

print(
    f"💼 Job Segment    : "
    f"{best_job} ({best_job_rate:.2f}%)"
)

print(
    f"👤 Age Group      : "
    f"{best_age} ({best_age_rate:.2f}%)"
)

print(
    f"📚 Education      : "
    f"{best_education} ({best_education_rate:.2f}%)"
)

print(
    f"🔄 Previous Campaign Outcome : "
    f"{best_previous} ({best_previous_rate:.2f}%)"
)


# ================================================================
# BUSINESS RECOMMENDATIONS
# ================================================================

print("\n")
print("=" * 80)
print("💡 BUSINESS RECOMMENDATIONS")
print("=" * 80)

print("""
1️⃣ Focus marketing efforts on customer segments with stronger
   historical conversion rates.

2️⃣ Analyze the performance of different contact channels and
   allocate campaign resources based on conversion results.

3️⃣ Personalize marketing messages according to customer
   characteristics such as age, job and education.

4️⃣ Monitor campaign frequency and avoid unnecessary repeated
   customer contact.

5️⃣ Use previous campaign outcomes to identify customers who
   may be more receptive to future campaigns.

6️⃣ Track conversion and drop-off rates regularly to identify
   opportunities for funnel optimization.

7️⃣ Continue testing different campaign strategies and compare
   their conversion performance over time.
""")


# ================================================================
# SAVE OUTPUT FILES
# ================================================================

df.to_csv(
    "Future_Interns_Task_3_Cleaned_Data.csv",
    index=False
)

channel_data.to_csv(
    "Task3_Channel_Conversion.csv"
)

age_data.to_csv(
    "Task3_Age_Conversion.csv"
)

job_data.to_csv(
    "Task3_Job_Conversion.csv"
)

education_data.to_csv(
    "Task3_Education_Conversion.csv"
)

month_data.to_csv(
    "Task3_Month_Conversion.csv"
)

previous_data.to_csv(
    "Task3_Previous_Campaign_Conversion.csv"
)


# ================================================================
# FINAL MESSAGE
# ================================================================

print("\n")
print("=" * 80)
print("🎉 FUTURE INTERNS TASK 3 COMPLETED SUCCESSFULLY!")
print("=" * 80)

print("""
✅ Dataset downloaded
✅ Dataset extracted
✅ Dataset cleaned
✅ Marketing funnel calculated
✅ Conversion rate calculated
✅ Drop-off rate calculated
✅ Contact channel analysis
✅ Age group analysis
✅ Job analysis
✅ Education analysis
✅ Monthly analysis
✅ Previous campaign analysis
✅ Colorful interactive dashboard created
✅ Key insights generated
✅ Business recommendations generated
✅ Output CSV files saved
""")

print("=" * 80)
print("🌈 Your Task 3 dashboard is ready!")
print("=" * 80)

🚀 Starting Future Interns Task 3...
✅ Dataset downloaded successfully!
✅ Dataset extracted successfully!

📂 CSV files found:
    future_interns_task3_data/bank-full.csv
    future_interns_task3_data/bank.csv

✅ Dataset selected:
future_interns_task3_data/bank-full.csv

✅ Dataset loaded successfully!
Rows: 45211
Columns: 17
✅ Data cleaning completed!




📊 FUTURE INTERNS TASK 3 - KEY INSIGHTS

👥 Total Marketing Contacts : 45,211
🎯 Successful Conversions   : 5,289
📈 Conversion Rate          : 11.70%
📉 Drop-off Rate            : 88.30%

🏆 TOP PERFORMING SEGMENTS
--------------------------------------------------------------------------------
📱 Contact Channel : cellular (14.92%)
💼 Job Segment    : student (28.68%)
👤 Age Group      : 66+ (42.61%)
📚 Education      : tertiary (15.01%)
🔄 Previous Campaign Outcome : success (64.73%)


💡 BUSINESS RECOMMENDATIONS

1️⃣ Focus marketing efforts on customer segments with stronger
   historical conversion rates.

2️⃣ Analyze the performance of different contact channels and
   allocate campaign resources based on conversion results.

3️⃣ Personalize marketing messages according to customer
   characteristics such as age, job and education.

4️⃣ Monitor campaign frequency and avoid unnecessary repeated
   customer contact.

5️⃣ Use previous campaign outcomes to identify customers who
   may be more